# Agent's side  $\rho$ (Inner Optimization)


## Single agent

### Objective of the Agent :
The optimal policy in the perspective of the agent is the best responding to $\rho$.  The agent is guided by the decisions of the principal made accordingly to the outcomes.

This is done by contracts: the agent would originally choose the action requesting less effort, but he is offered a contract $b$ by the pricipal prior to taking his choice of action. This incentivate the agent to behave accondingly to the contractual payment (non-negative values). 


### Elements of agent's perspective

- set of actions: $A$
- set of costs: $c(a)$ , interpreted as effort levels
- set of outcome: $O$
- contractual payments $b(o)$ , dependant on outcome
- set of states: $S^{a} = S \times B$
- set of initial states: $S_0 = \{s_{0}\} \times  B$
- distribution regulating outcomes ( with higher effort
levels more likely to result in rewarding outcomes)
- agent's reward: $r(s,a)+b(o)$, with $𝑟(𝑠,𝑎) = -c(a)$


### Q-learning functions
##### Q-function \( Q(s,b,a) \)

The Q-function represents the expected discounted utility of the agent when taking action \( a \) in state \( s \) under contract \( b \), and then following the optimal policy thereafter.

$$
Q(s,b,a) =
\mathbb{E}\left[
r(s,a) + b(o) + \gamma \max_{a'} Q(s', b', a')
\right]
$$

where:
- \( $o \sim O(s,a)$ \) is the outcome,
- \( $s' \sim P(s' \mid s,a)$ \) is the next state,
- \( $b' \sim \rho(s')$ \) is the next contract offered by the principal.

**Interpretation:**  
This function captures how attractive an action is **given a specific contract**, taking into account both:
- immediate reward (effort cost + payment),
- future contracts induced by the principal’s policy \( \rho \).

---

##### Truncated Q-function \( $\overline{Q}(s,a)$ \)

The truncated Q-function marginalizes over contracts and represents the expected value of taking action \( a \) in state \( s \), under the distribution of contracts induced by the principal’s policy.

$$
\overline{Q}(s,a)
=
\mathbb{E}_{b \sim \rho(\cdot \mid s)}\left[ Q(s,b,a) \right]
$$

**Interpretation:**  
This quantity reflects how the agent evaluates an action **on average**, given how the principal typically behaves.

- It removes the explicit dependence on contracts  
- It is the object used by the principal when designing incentive-compatible contracts  

---

### Role in the Model

- \( $Q(s,b,a)$ \): used by the agent to determine its optimal policy  
  $$
  \pi(s,b) = \arg\max_a Q(s,b,a)
  $$

- ( $\overline{Q}(s,a)$ ): used by the principal to enforce incentive constraints in contract desig. Eventhough we don't use it directly in the agent's MDP, we calculate it to store it for the principal having the whole system in mind.



Inner optimisation code using Q-learning algorithm and defined values for contracts :

In [3]:
import numpy as np

class Agent:

    '''
    initialisation of MDP with needed parameters and serialisation of contracts
    '''
    def __init__(self, mdp, alpha=0.1, epsilon=0.1, b_grid_step=0.1):
        self.mdp = mdp
        self.n_states = mdp.n_states
        self.n_actions = mdp.n_actions
        self.n_outcomes = mdp.n_outcomes

        self.alpha = alpha
        self.gamma = mdp.gamma
        self.epsilon = epsilon

        self.cost = np.array([0.1, 0.5])  # effort costs

        self.b_values = np.round(np.arange(0, 1.01, b_grid_step), 3)
        self.b_grid = self._build_contract_grid()

        self.q = np.zeros((self.n_states, len(self.b_grid), self.n_actions))


    def _build_contract_grid(self):
        grid = []
        for bL in self.b_values:
            for bR in self.b_values:
                grid.append((bL, bR))
        return grid

    def contract_to_id(self, b):
        return self.b_grid.index(tuple(np.round(b, 3)))


    '''
    makes decision of action accordingly to state and contract, policy
    '''
    def act(self, state, b):
        b_id = self.contract_to_id(b)

        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)

        return np.argmax(self.q[state, b_id])

    '''
    returns rewards as benefice of contract minus costs of action
    '''
    def compute_reward(self, s, a, o, b):
        return -self.cost[a] + b[o]

    def update(self, s, b, a, o, s_next, rho):
        r_agent = self.compute_reward(s, a, o, b)

        b_id = self.contract_to_id(b)

        b_next = rho(s_next)
        b_next_id = self.contract_to_id(b_next)

        max_next = np.max(self.q[s_next, b_next_id])

        # adapted Q-learning with contracts
        target = r_agent + self.gamma * max_next
        self.q[s, b_id, a] += self.alpha * (target - self.q[s, b_id, a])

    '''
    calculate the value of Q_bar
    '''
    def get_Q_bar(self, rho):
        Q_bar = np.zeros((self.n_states, self.n_actions))

        for s in range(self.n_states):
            b = rho(s)
            b_id = self.contract_to_id(b)
            Q_bar[s] = self.q[s, b_id]

        return Q_bar
    

    def reset(self):
        self.q = np.zeros_like(self.q)

UPDATE AFTER MEETING:
work done without learning, based on the following formulas:

![image](images/first_algo_functions.png)

## Agent's side
- has set of actions $A$ from which he chose one accordingly to the situation (state and contract + policy)
- observes pair $(s_t, b_t)$, $s_t$ being the state and $b_t$ the contract proposed by the principal
- makes decisions based on his policy: $ \pi : S \times B \rightarrow \Delta (A)$
- utility: value in the initial state $V^{\pi}(s_0, b_0 \mid \rho)$

The agent wants to maximize his *value function*, $V^{\pi}(\rho)$, the function is defined in a state $s$ given a contract $b$:

$V^{\pi}(s, b \mid \rho) = \mathbb{E}\left[ \sum_{t} \gamma^{t} R(s_t, a_t, b_t, o_t) \,\middle|\, s_0 = s, b_0 = b \right]$

In this meta-algorithm, the Q-value function used by the agent can be defined as:

$Q^{\pi}((s,b), a \mid \rho) = V^{\pi}(s, b \mid a_0 = a, \rho)$

The goal is to choose the optimal policy $\pi^{*}$

In [ ]:
import numpy as np

class Agent:
    """
    Agent in a Principal-Agent MDP.
    Solves the inner optimization of the meta-algorithm:
    given principal's policy rho, find best-responding pi*.
    """
    def __init__(self, mdp):
        self.mdp       = mdp
        self.n_states  = mdp.n_states
        self.n_actions = mdp.n_actions
        self.n_outcomes = mdp.n_outcomes
        self.gamma     = mdp.gamma
  

        # these are set after solve() is called
        self.V       = None   # V_agent[s]
        self.Q_agent = None   # Q^pi((s,b), a | rho)
        self.pi_star = None   # pi*(s, b) -> action

    def solve(self, rho, tol=1e-10, max_iter=1000):
        """
        Fix rho: s -> np.array([b(L), b(R)]) and solve agent's MDP
        via value iteration.

        Sets self.V, self.Q_agent, self.pi_star
        """
        mdp = self.mdp
        nS  = self.n_states
        nA  = self.n_actions
        V   = np.zeros(nS)

        # value iteration
        for _ in range(max_iter):
            V_new = np.zeros(nS)
            for s in range(nS):
                b = rho[s]
                Q = np.zeros(nA)
                for a in range(nA):
                    for o in range(mdp.n_outcomes):
                        p      = mdp.P_outcome[s, a, o]
                        s_next = mdp.T(s, o)
                        Q[a]  += p * (mdp.R_agent(s, a, b, o) + self.gamma * V[s_next])
                V_new[s] = np.max(Q)

            if np.max(np.abs(V - V_new)) < tol:
                break
            V = V_new

        self.V = V

        # Q^pi((s,b), a | rho) for arbitrary b at current state
        # future states still use rho(s') — captured in V
        def Q_agent(s, b, a):
            q = 0.0
            for o in range(mdp.n_outcomes):
                p      = mdp.P_outcome[s, a, o]
                s_next = mdp.T(s, o)
                q     += p * (mdp.R_agent(s, a, b, o) + self.gamma * V[s_next])
            return q

        # pi*(s, b) = argmax_a Q^pi((s,b), a)
        # ties broken in favor of principal (principal Q injected later if needed)
        def pi_star(s, b):
            Q_vals  = np.array([Q_agent(s, b, a) for a in range(nA)])
            max_val = np.max(Q_vals)
            tied    = np.where(np.abs(Q_vals - max_val) < 1e-9)[0]
            return tied[0]

        self.Q_agent = Q_agent
        self.pi_star = pi_star

        return V, Q_agent, pi_star
    